In [3]:
## Imports
import numpy as np
import pandas as pd
!pip install catboost
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

In [4]:
## Load Datasets
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

In [5]:
## Check Shape
print("train shape:",train.shape)
print("public_test shape",public_test.shape)
print("private_test shape",private_test.shape)

train shape: (10000, 14)
public_test shape (3000, 14)
private_test shape (3000, 13)


In [6]:
## See training data
train.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


In [7]:
## Dataset Information
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  int64  
 1   Age                 8520 non-null   float64
 2   Income              9016 non-null   float64
 3   City_Tier           10000 non-null  int64  
 4   Device_Type         10000 non-null  object 
 5   Traffic_Source      10000 non-null  object 
 6   Pages_Viewed        10000 non-null  int64  
 7   Products_Viewed     10000 non-null  int64  
 8   Time_On_Site        8152 non-null   float64
 9   Previous_Purchases  10000 non-null  int64  
 10  Discount_Seen       10000 non-null  int64  
 11  Browser_Version     10000 non-null  int64  
 12  Campaign_Code       10000 non-null  int64  
 13  Converted           10000 non-null  int64  
dtypes: float64(3), int64(9), object(2)
memory usage: 1.1+ MB


In [8]:
## Missing values
train.isnull().sum().sort_values(ascending=False)

,0
Time_On_Site,1848
Age,1480
Income,984
User_ID,0
Device_Type,0
City_Tier,0
Traffic_Source,0
Pages_Viewed,0
Products_Viewed,0
Previous_Purchases,0


In [9]:
## Understand Target Distribution
train['Converted'].value_counts()
train['Converted'].value_counts(normalize=True) * 100

,proportion
Converted,
0,69.13
1,30.87


In [10]:
## EDA
num_cols=[
    'Age','Income','Time_On_Site','Products_Viewed','Pages_Viewed', 'Previous_Purchases'
]
for col in num_cols:
    print("\n" + "="*50)
    print(col)
    print(train.groupby("Converted")[col].mean())


Age
Converted
0    41.548475
1    41.253435
Name: Age, dtype: float64

Income
Converted
0    69134.659095
1    71771.595850
Name: Income, dtype: float64

Time_On_Site
Converted
0    12.597556
1    15.925828
Name: Time_On_Site, dtype: float64

Products_Viewed
Converted
0    13.360480
1    19.381924
Name: Products_Viewed, dtype: float64

Pages_Viewed
Converted
0    13.802546
1    19.652737
Name: Pages_Viewed, dtype: float64

Previous_Purchases
Converted
0    2.859974
1    3.224490
Name: Previous_Purchases, dtype: float64


In [11]:
cat_cols = [
    'City_Tier',
    'Device_Type',
    'Traffic_Source',
    'Discount_Seen',
    'Browser_Version',
    'Campaign_Code'
]

for col in cat_cols:
    print("\n" + "="*50)
    print(col)
    print(pd.crosstab(train[col], train['Converted'], normalize='index') * 100)


City_Tier
Converted          0          1
City_Tier                      
1          69.700910  30.299090
2          67.966327  32.033673
3          70.580913  29.419087

Device_Type
Converted            0          1
Device_Type                      
Desktop      66.909385  33.090615
Mobile       69.937145  30.062855
Tablet       69.353234  30.646766

Traffic_Source
Converted               0          1
Traffic_Source                      
Email           65.064103  34.935897
Organic         70.823836  29.176164
Paid Ads        72.287736  27.712264
Referral        61.881868  38.118132
Social Media    69.693929  30.306071

Discount_Seen
Converted              0          1
Discount_Seen                      
0              74.482909  25.517091
1              64.582948  35.417052

Browser_Version
Converted                0          1
Browser_Version                      
1                69.841270  30.158730
2                70.997680  29.002320
3                71.394231  28.605769
4    

In [16]:
missing_cols = [
    'Age',
    'Income',
    'Time_On_Site'
]

for col in missing_cols:

    median_value = train[col].median()

    train[col] = train[col].fillna(median_value)

    public_test[col] = public_test[col].fillna(median_value)

    private_test[col] = private_test[col].fillna(median_value)

In [17]:
X_train = train.drop(
    ["User_ID", "Converted"],
    axis=1
)

y_train = train["Converted"]

X_valid = public_test.drop(
    ["User_ID", "Converted"],
    axis=1
)

y_valid = public_test["Converted"]

In [18]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='F1',
    random_state=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

0:	learn: 0.2384378	total: 71.8ms	remaining: 1m 11s
100:	learn: 0.3922356	total: 2.4s	remaining: 21.4s
200:	learn: 0.4607574	total: 4.4s	remaining: 17.5s
300:	learn: 0.5119934	total: 6.62s	remaining: 15.4s
400:	learn: 0.5482358	total: 8.91s	remaining: 13.3s
500:	learn: 0.5819357	total: 12.2s	remaining: 12.1s
600:	learn: 0.6114906	total: 14.4s	remaining: 9.56s
700:	learn: 0.6355476	total: 16.6s	remaining: 7.1s
800:	learn: 0.6562003	total: 18.9s	remaining: 4.69s
900:	learn: 0.6825054	total: 21.1s	remaining: 2.32s
999:	learn: 0.7076684	total: 24.5s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='F1', iterations=1000, learning_rate=0.05, random_state=42, verbose=100)

In [19]:
y_pred = model.predict(X_valid)

In [20]:
print("Accuracy :", accuracy_score(y_valid, y_pred))

print("F1 Score :", f1_score(y_valid, y_pred))

print("\nClassification Report\n")

print(classification_report(y_valid, y_pred))

Accuracy : 0.724
F1 Score : 0.4242002781641168

Classification Report

              precision    recall  f1-score   support

           0       0.76      0.88      0.82      2114
           1       0.55      0.34      0.42       886

    accuracy                           0.72      3000
   macro avg       0.66      0.61      0.62      3000
weighted avg       0.70      0.72      0.70      3000



In [21]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance)

               Feature  Importance
7         Time_On_Site   13.397905
10     Browser_Version   13.044681
1               Income   12.492878
6      Products_Viewed   11.432312
4       Traffic_Source    8.844545
5         Pages_Viewed    8.834806
0                  Age    6.832846
2            City_Tier    5.915243
3          Device_Type    5.866017
8   Previous_Purchases    5.633648
11       Campaign_Code    4.355133
9        Discount_Seen    3.349986


In [22]:
## feature Engineering
X_train['Engagement'] = (
    X_train['Pages_Viewed'] *
    X_train['Time_On_Site']
)

X_valid['Engagement'] = (
    X_valid['Pages_Viewed'] *
    X_valid['Time_On_Site']
)

In [23]:
X_train['Interest_Ratio'] = (
    X_train['Products_Viewed'] /
    (X_train['Pages_Viewed'] + 1)
)

X_valid['Interest_Ratio'] = (
    X_valid['Products_Viewed'] /
    (X_valid['Pages_Viewed'] + 1)
)

In [24]:
X_train['Returning_Customer'] = (
    X_train['Previous_Purchases'] > 0
).astype(int)

X_valid['Returning_Customer'] = (
    X_valid['Previous_Purchases'] > 0
).astype(int)

In [26]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='F1',
    random_state=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

0:	learn: 0.3833959	total: 172ms	remaining: 2m 52s
100:	learn: 0.4113938	total: 4.63s	remaining: 41.2s
200:	learn: 0.4748592	total: 7.72s	remaining: 30.7s
300:	learn: 0.5214183	total: 10.1s	remaining: 23.4s
400:	learn: 0.5551471	total: 13.5s	remaining: 20.1s
500:	learn: 0.5883306	total: 15.8s	remaining: 15.8s
600:	learn: 0.6156011	total: 18.1s	remaining: 12s
700:	learn: 0.6399681	total: 20.4s	remaining: 8.71s
800:	learn: 0.6648188	total: 22.8s	remaining: 5.67s
900:	learn: 0.6887755	total: 26.2s	remaining: 2.88s
999:	learn: 0.7107662	total: 28.6s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='F1', iterations=1000, learning_rate=0.05, random_state=42, verbose=100)

In [27]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = model.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, y_pred))
print("F1 Score:", f1_score(y_valid, y_pred))

Accuracy: 0.7256666666666667
F1 Score: 0.4248777078965758


In [28]:
from sklearn.metrics import f1_score
import numpy as np

probs = model.predict_proba(X_valid)[:, 1]

best_f1 = 0
best_threshold = 0

for threshold in np.arange(0.1, 0.9, 0.01):

    preds = (probs >= threshold).astype(int)

    score = f1_score(y_valid, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

Best Threshold: 0.2599999999999999
Best F1 Score: 0.5530884134033105


In [29]:
## Full Final Code

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test = pd.read_csv("private_test.csv")

full_train = pd.concat(
    [train, public_test],
    ignore_index=True
)

missing_cols = [
    'Age',
    'Income',
    'Time_On_Site'
]

for col in missing_cols:

    median_value = full_train[col].median()

    full_train[col] = full_train[col].fillna(median_value)

    private_test[col] = private_test[col].fillna(median_value)

full_train['Engagement'] = (
    full_train['Pages_Viewed']
    * full_train['Time_On_Site']
)

private_test['Engagement'] = (
    private_test['Pages_Viewed']
    * private_test['Time_On_Site']
)

full_train['Interest_Ratio'] = (
    full_train['Products_Viewed']
    /
    (full_train['Pages_Viewed'] + 1)
)

private_test['Interest_Ratio'] = (
    private_test['Products_Viewed']
    /
    (private_test['Pages_Viewed'] + 1)
)

full_train['Returning_Customer'] = (
    full_train['Previous_Purchases'] > 0
).astype(int)

private_test['Returning_Customer'] = (
    private_test['Previous_Purchases'] > 0
).astype(int)


X_train = full_train.drop(
    ["User_ID", "Converted"],
    axis=1
)

y_train = full_train["Converted"]

X_private = private_test.drop(
    ["User_ID"],
    axis=1
)


cat_features = [
    'City_Tier',
    'Device_Type',
    'Traffic_Source',
    'Discount_Seen',
    'Browser_Version',
    'Campaign_Code'
]


model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='F1',
    random_state=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

probs = model.predict_proba(X_private)[:, 1]

best_threshold = 0.26

predictions = (
    probs >= best_threshold
).astype(int)

# CREATE SUBMISSION

submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully!")
print(submission.head())

0:	learn: 0.2001210	total: 26.2ms	remaining: 26.2s
100:	learn: 0.3949811	total: 5.28s	remaining: 47s
200:	learn: 0.4503441	total: 8.81s	remaining: 35s
300:	learn: 0.4908294	total: 11.6s	remaining: 27s
400:	learn: 0.5187570	total: 14.4s	remaining: 21.5s
500:	learn: 0.5426282	total: 17.2s	remaining: 17.1s
600:	learn: 0.5680000	total: 21.3s	remaining: 14.2s
700:	learn: 0.5888341	total: 24.2s	remaining: 10.3s
800:	learn: 0.6080783	total: 27s	remaining: 6.72s
900:	learn: 0.6263891	total: 29.9s	remaining: 3.29s
999:	learn: 0.6435705	total: 33.9s	remaining: 0us
submission.csv created successfully!
   User_ID  Converted
0   103001          0
1   103002          0
2   103003          1
3   103004          1
4   103005          0


In [30]:
import os
print(os.path.abspath("submission.csv"))

/content/submission.csv
